# 13 — Unified Customer Profile & 360-Degree Intelligence
Combining customer segmentation, churn probability, and inactivity monitoring into a single unified analytical profile for the executive dashboard.

In [1]:
import pandas as pd, numpy as np, sqlite3, joblib, os
import pyarrow as pa, pyarrow.parquet as pq
import plotly.express as px, plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load and Merge Datasets ──
cust = pd.read_csv('../../data/processed/customer_with_segments.csv')
inact = pd.read_csv('../../data/processed/inactivity_scores.csv')

profile = cust.merge(
    inact[['CLIENTNUM','activity_score','activity_category','future_churn_candidate']],
    on='CLIENTNUM', how='left'
)

# For churn: re-predict on full dataset using saved XGBoost model
xgb_model = joblib.load('../../models/churn/xgboost_model.pkl')
le_income = joblib.load('../../models/churn/le_income.pkl')
le_card = joblib.load('../../models/churn/le_card.pkl')

profile['Income_Category_enc'] = le_income.transform(profile['Income_Category'])
profile['Card_Category_enc'] = le_card.transform(profile['Card_Category'])

feat_cols = ['Customer_Age','Income_Category_enc','Card_Category_enc','Months_on_book',
             'Months_Inactive_12_mon','Contacts_Count_12_mon','Credit_Limit',
             'Total_Trans_Amt','Total_Trans_Ct','Avg_Utilization_Ratio']

profile['churn_probability'] = xgb_model.predict_proba(profile[feat_cols])[:,1]
profile['risk_label'] = pd.cut(
    profile['churn_probability'],
    bins=[-0.001, 0.40, 0.70, 1.001],
    labels=['Loyal', 'Medium Risk', 'High Risk']
)
print(f"Merged profile shape: {profile.shape}")

Merged profile shape: (10127, 33)


In [3]:
# ── SQL Executive KPIs ──
con = sqlite3.connect(':memory:')
profile.to_sql('profile', con, index=False, if_exists='replace')

q1 = pd.read_sql_query("""
    SELECT
      COUNT(*) as total_customers,
      ROUND(AVG(is_attrited)*100,2) as churn_rate_pct,
      SUM(CASE WHEN risk_label='High Risk' THEN 1 ELSE 0 END) as high_risk_count,
      ROUND(AVG(churn_probability),4) as avg_churn_prob,
      SUM(future_churn_candidate) as watchlist_count,
      ROUND(AVG(Credit_Limit),2) as avg_credit_limit,
      SUM(CASE WHEN activity_category='Active' THEN 1 ELSE 0 END) as active_customers
    FROM profile
""", con)

print("=== EXECUTIVE KPIs (ALL REAL FROM DATA) ===")
print(q1.to_string(index=False))

q2 = pd.read_sql_query("""
    SELECT segment, COUNT(*) as count,
           ROUND(AVG(churn_probability)*100,2) as avg_churn_pct,
           ROUND(AVG(activity_score),4) as avg_activity,
           SUM(future_churn_candidate) as watchlist
    FROM profile GROUP BY segment ORDER BY avg_churn_pct DESC
""", con)

print("\n=== SEGMENT SUMMARY FOR FRONTEND ===")
print(q2.to_string(index=False))

=== EXECUTIVE KPIs (ALL REAL FROM DATA) ===
 total_customers  churn_rate_pct  high_risk_count  avg_churn_prob  watchlist_count  avg_credit_limit  active_customers
           10127         16.0700             1586          0.1640              135         8631.9500               253

=== SEGMENT SUMMARY FOR FRONTEND ===
          segment  count  avg_churn_pct  avg_activity  watchlist
At-Risk Customers   2451        30.6900        0.3177        106
Premium Customers   1194        18.4000        0.3834         18
     Silent Users   2730        15.7900        0.4282          0
     Deal Hunters   2883         8.3500        0.4538         11
   Daily Spenders    869         1.9500        0.6524          0


In [4]:
# ── 360-Degree Executive Visualizations ──
# Plot 1: Segment Churn Risk Summary
fig1 = px.bar(q2, x='segment', y='avg_churn_pct', color='watchlist', title="Average Churn Risk by Customer Segment (with Watchlist Count)")
fig1.show()

# Plot 2: Risk Label vs Activity Category Heatmap
xtab = pd.crosstab(profile['risk_label'], profile['activity_category'])
fig2 = px.imshow(xtab, text_auto=True, title="Customer Risk Label vs Activity Category Matrix")
fig2.show()

# Plot 3: Churn Probability Distribution by Segment
fig3 = px.box(profile, x='segment', y='churn_probability', color='segment', title="Churn Probability Distribution across Segments")
fig3.show()

# Plot 4: Credit Limit vs Total Trans Amount colored by Risk Label
sample_plot = profile.sample(1500, random_state=42)
fig4 = px.scatter(sample_plot, x='Credit_Limit', y='Total_Trans_Amt', color='risk_label', size='churn_probability', title="Credit Limit vs Spend Volume by Risk Level (Sample 1500)", opacity=0.7)
fig4.show()

In [5]:
# ── Save Unified Profile ──
os.makedirs('../../data/features', exist_ok=True)
profile.to_csv('../../data/processed/unified_customer_profile.csv', index=False)
pa.parquet.write_table(pa.Table.from_pandas(profile), '../../data/features/customer_features.parquet')

print("=== UNIFIED PROFILE SAVED ===")
print(f"Total customers: {len(profile)}")
print(f"High Risk: {(profile['risk_label']=='High Risk').sum()}")
print(f"Churn Rate: {profile['is_attrited'].mean()*100:.2f}%")
print(f"Active: {(profile['activity_category']=='Active').sum()}")
print(f"Watchlist: {profile['future_churn_candidate'].sum()}")
print("\nThese are the REAL numbers that go into the frontend dashboard.")
con.close()

=== UNIFIED PROFILE SAVED ===
Total customers: 10127
High Risk: 1586
Churn Rate: 16.07%
Active: 253
Watchlist: 135

These are the REAL numbers that go into the frontend dashboard.
